# 疾患メカニズムに関わる遺伝子を順位づける（設定は YAML）

2 万遺伝子から出発し、3 段階で絞り込みます。**疾患と候補遺伝子はこのノートブックの ① に書き、
それ以外の設定（モデル、各段階の遺伝子数、質問文、表示、見本、保存先）は `gene_ranking_config.yaml` に書きます。**

| 段階 | やること | 既定の規模 | 読み取るもの |
|---|---|---|---|
| 0 | 二値（Yes / No） | 19,297 → 2,000 | `logP(Yes) − logP(No)` |
| 1 | 選択式（群 5 ＋「その他」） | 2,000 → 600 → 200 | 選択肢ごとの確率 |
| 2 | 総当たり（1 対 1 ＋「どちらも関係ない」、両方向） | 200 → 順位 | 同上 → Bradley-Terry |

**どの段階も問いの考え方は同じです。**疾患メカニズムを箇条書きで示し、
「この遺伝子の活性を上げても下げても、これらの段階のどれか（any of these steps）が変わるか」を聞きます。
「治療標的か」とは聞きません。既存の治療手段に引きずられて、低分子薬の標的ばかりが上に来るためです。

**モデルに遺伝子記号を書かせることはしません。**Yes / No や A〜F の 1 トークンの確率だけを読み、
記号への対応はコードが持ちます。記号は複数トークンに分かれるので、書かせると GPR52 と GPR56 のような
似た記号を取り違えます。

## 0. 設定を読む

In [ ]:
import csv, itertools, json, math, os, random, shutil, statistics, string, time
import urllib.error, urllib.request
from collections import Counter, defaultdict

import yaml

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
    print("pandas がありません:  pip install pandas （表は簡易表示になります）")

# 設定ファイル。値はすべてこの中に書く。別の設定で回すときは環境変数で差し替える。
CONFIG_PATH = os.environ.get("GENE_RANKING_CONFIG", "gene_ranking_config.yaml")
with open(CONFIG_PATH) as fh:
    CFG = yaml.safe_load(fh)


def check_config(cfg):
    """よくある書き間違いを、実行が何時間も進んでから気づくのではなく最初に止める。"""
    errs = []
    for key in ("models", "stage0", "stage1", "stage2", "display", "rating", "few_shot", "output"):
        if key not in cfg:
            errs.append(f"`{key}` がありません")
    if errs:
        raise ValueError("設定の不備: " + " / ".join(errs))
    gs = cfg["stage1"]["group_size"]
    if not 2 <= gs <= 25:
        errs.append(f"stage1.group_size は 2〜25（いまは {gs}）。「その他」を含めて 26 文字に収める")
    keeps = [r["keep"] for r in cfg["stage1"]["rounds"]]
    if any(b >= a for a, b in zip(keeps, keeps[1:])):
        errs.append(f"stage1.rounds の keep は減っていく必要があります（いまは {keeps}）")
    if cfg["stage0"]["keep"] < keeps[0]:
        errs.append(f"stage0.keep（{cfg['stage0']['keep']}）が stage1 の最初の keep（{keeps[0]}）より少ない")
    if errs:
        raise ValueError("設定の不備:\n  " + "\n  ".join(errs))


check_config(CFG)


def normalize_host(h):
    """OLLAMA_HOST を URL に直す。Ollama のクライアントと同じ読み方にする。

    Ollama は `0.0.0.0:11434` や `ollama:11434` のようにスキーム無しで書く（Docker の設定でよく見る）。
    そのまま urllib に渡すと 'ollama' をスキームと読んで失敗し、サーバーが落ちているように見える。
    スキーム無しならポート 11434、http:// や https:// を明示したらそのスキームの既定ポート。
    0.0.0.0 は待ち受けのアドレスなので、接続先としては localhost に読み替える。"""
    h = (h or "").strip().rstrip("/") or "localhost:11434"
    had_scheme = "://" in h
    if not had_scheme:
        h = "http://" + h
    scheme, rest = h.split("://", 1)
    hostport = rest.split("/", 1)[0]
    if hostport.startswith("0.0.0.0"):
        hostport = "localhost" + hostport[len("0.0.0.0"):]
    if not had_scheme and ":" not in hostport.rsplit("]", 1)[-1]:
        hostport += ":11434"
    return f"{scheme}://{hostport}"


OLLAMA_HOST = normalize_host(CFG["ollama"].get("host") or os.environ.get("OLLAMA_HOST"))
print(f"設定  : {CONFIG_PATH}")
print(f"Ollama: {OLLAMA_HOST}")

In [ ]:
# ===== 保存の道具 =====
# 1 回の実行につき outputs/日付-時刻/ を 1 つ作り、段階ごとに生データを置く。
# 全体で数時間かかるので、途中で落ちても前の段階をやり直さずに済むようにする。
RUN_DIR = None


def new_run_dir(root):
    base = os.path.join(root, time.strftime("%Y%m%d-%H%M%S"))
    d, k = base, 1
    while os.path.exists(d):          # 同じ秒に 2 回走っても衝突しない
        d = f"{base}_{k}"
        k += 1
    os.makedirs(d)
    return d


def run_file(name):
    if RUN_DIR:
        p = os.path.join(RUN_DIR, name)
        if os.path.exists(p):
            return p
    return None


def save_json(name, obj):
    if not RUN_DIR:
        return None
    p = os.path.join(RUN_DIR, name)
    with open(p, "w") as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=1)
    print(f"  保存: {p}")
    return p


def save_rows(name, header, rows):
    if not RUN_DIR:
        return None
    p = os.path.join(RUN_DIR, name)
    with open(p, "w", newline="") as fh:
        w = csv.writer(fh, delimiter="\t")
        w.writerow(header)
        w.writerows(rows)
    print(f"  保存: {p}（{len(rows)} 行）")
    return p


def load_rows(name):
    with open(os.path.join(RUN_DIR, name), newline="") as fh:
        return list(csv.DictReader(fh, delimiter="\t"))


def read_meta():
    p = run_file("run.json")
    if not p:
        return {}
    with open(p) as fh:
        return json.load(fh)


def record_meta(**kw):
    """run.json を段階が終わるたびに更新する。最後にまとめて書くと、途中で落ちたとき何も残らない。"""
    if not RUN_DIR:
        return
    meta = read_meta()
    meta.update(kw)
    with open(os.path.join(RUN_DIR, "run.json"), "w") as fh:
        json.dump(meta, fh, ensure_ascii=False, indent=1)

## ① 疾患と候補遺伝子

**ここだけは YAML ではなくノートブックに書きます。**

`DISEASE_MECHANISM` の各段階は「**変えられる要因**」を主語にして書いてください。
実測では「腎臓が尿を濃縮するほど、尿中のシスチン濃度が上がる」と書くと AVPR2（尿を濃くするホルモンの受容体）に届き、
「シスチンが尿中で濃縮して結石になる」と書くと届きませんでした（logit 差 −30 の No）。

In [ ]:
def load_genes(path):
    """記号<TAB>HGNC正式名<TAB>UniProt推奨名<TAB>別名（カンマ区切り）を読む。2〜4 列目は任意。"""
    seen, genes, names, proteins, aliases, dupes = set(), [], {}, {}, {}, 0
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.split("\t")
            g = parts[0].strip()
            if not g or g in seen:
                dupes += g in seen
                continue
            seen.add(g)
            genes.append(g)
            if len(parts) > 1 and parts[1].strip():
                names[g] = parts[1].strip()
            if len(parts) > 2 and parts[2].strip():
                proteins[g] = parts[2].strip()
            if len(parts) > 3 and parts[3].strip():
                aliases[g] = [a.strip() for a in parts[3].split(",") if a.strip()]
    return genes, names, proteins, aliases, dupes


# ===== 疾患。プロンプトにこのまま入る =====
DISEASE = "cystinuria"

# ===== 疾患メカニズム（段階の箇条書き）=====
DISEASE_MECHANISM = [
    "The kidney tubule cannot reabsorb cystine.",
    "The more the kidney concentrates the urine, the higher the cystine concentration in the urine.",
    "Cystine crystallizes more easily when the urine is acidic.",
]

# ===== 候補遺伝子のファイル =====
#   記号<TAB>HGNC正式名<TAB>UniProt推奨タンパク質名<TAB>別名（カンマ区切り）。2〜4 列目は任意
GENE_FILE = "genelist_cystinuria_mech1000.txt"

# ===== 答え合わせ用（任意）=====
KNOWN_ANSWERS = ["SLC3A1", "SLC7A9",   # 原因遺伝子
                 "SLC5A2", "AVPR2"]    # 臨床で検討中（dapagliflozin NCT04818034、tolvaptan NCT02538016）

if not DISEASE_MECHANISM:
    raise ValueError("DISEASE_MECHANISM が空です（段階を 1 行以上）")
MECHANISM = list(DISEASE_MECHANISM)
GENES, GENE_NAMES, PROTEIN_NAMES, ALIASES, _dupes = load_genes(GENE_FILE)

print(f"疾患  : {DISEASE}")
print("メカニズム:")
for _s in MECHANISM:
    print(f"  - {_s}")
print(f"候補  : {len(GENES)} 遺伝子（{GENE_FILE}）" + (f"、重複 {_dupes} 件を除外" if _dupes else ""))
print(f"蛋白質名 {len(PROTEIN_NAMES)} / HGNC 名 {len(GENE_NAMES)} / 別名あり {len(ALIASES)}")
_missing = [g for g in KNOWN_ANSWERS if g not in GENES]
print("答え合わせ:", KNOWN_ANSWERS, f"（候補に無い: {_missing}）" if _missing else "（いずれも候補に含まれる）")

# ===== 保存先（新規 or 再開）=====
RESUME_DIR = CFG["output"].get("resume_dir") or ""
if RESUME_DIR:
    if not os.path.isdir(RESUME_DIR):
        raise FileNotFoundError(f"{RESUME_DIR} がありません")
    RUN_DIR = RESUME_DIR
    _prev = read_meta()
    # 疾患・候補・メカニズムが違う実行に継ぎ足すと、別の問いの答えが混ざって結果が黙って壊れる
    for _key, _now in (("disease", DISEASE), ("gene_file", GENE_FILE), ("mechanism", MECHANISM)):
        if _prev.get(_key) not in (None, _now):
            raise ValueError(f"{RESUME_DIR} は {_key}={_prev[_key]!r} の実行です（いまの設定は {_now!r}）。")
    record_meta(resumed=_prev.get("resumed", []) + [time.strftime("%Y-%m-%d %H:%M:%S")])
    print(f"\n再開: {RUN_DIR}/  （{', '.join(sorted(os.listdir(RUN_DIR)))}）")
else:
    RUN_DIR = new_run_dir(CFG["output"]["root"])
    shutil.copy(CONFIG_PATH, os.path.join(RUN_DIR, "config.yaml"))   # その実行で使った設定を残す
    record_meta(started=time.strftime("%Y-%m-%d %H:%M:%S"), disease=DISEASE, mechanism=MECHANISM,
                gene_file=GENE_FILE, n_genes=len(GENES), known_answers=KNOWN_ANSWERS)
    print(f"\n保存先: {RUN_DIR}/")

## ② モデル

段階ごとのモデルは YAML の `models` で指定します。空欄は `default` を使います。

段階ごとに違うモデルを使う場合、VRAM に 2 つ載らないと呼び出しのたびに載せ替えが起き、
実測で 1 件 0.3 秒の処理が 30 秒になりました。各段階の開始前に、その段階で使わないモデルを降ろします
（Ollama サーバー本体は止めません）。

In [ ]:
def list_local_models():
    """Ollama に入っているモデル。埋め込み専用は logprobs を返さないので外す。"""
    try:
        with urllib.request.urlopen(OLLAMA_HOST + "/api/tags", timeout=10) as r:
            models = json.loads(r.read().decode()).get("models", [])
    except urllib.error.URLError as e:
        print(f"Ollama に接続できません: {OLLAMA_HOST}（{e.reason}）")
        print("  ・ローカルなら `ollama serve` が動いているか")
        print("  ・Docker なら `-p 11434:11434` でポートを出しているか")
        print("  ・Jupyter もコンテナ内なら localhost は自分自身を指す（http://ollama:11434 など）")
        return []
    skip = ("embed", "bge-", "e5-", "gte-")
    return [{"name": m["name"], "size_gb": round(m.get("size", 0) / 1e9, 1)}
            for m in models if not any(k in m["name"].lower() for k in skip)]


def resident_models():
    """いま Ollama に載っているモデル。[(名前, GB), ...]。"""
    try:
        with urllib.request.urlopen(OLLAMA_HOST + "/api/ps", timeout=10) as r:
            ms = json.loads(r.read().decode()).get("models", [])
    except urllib.error.URLError:
        return []
    return [(m.get("name", "?"), m.get("size", 0) / 1e9) for m in ms]


def unload(model):
    """そのモデルだけを VRAM から降ろす（`ollama stop <model>` と同じ）。サーバーは止めない。"""
    payload = {"model": model, "prompt": "", "keep_alive": 0}
    req = urllib.request.Request(OLLAMA_HOST + "/api/generate", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=60):
            print(f"  降ろしました: {model}")
    except urllib.error.URLError as e:
        print(f"  降ろせませんでした: {model}（{e}）")


def first_token_logprobs(prompt, model, top_logprobs=20, no_think=True):
    """次の 1 トークンの (生成トークン, {トークン: 対数確率})。Ollama の上限は 20。

    思考モデルは think:false で思考を止める。思考させると、段階0で既存薬の標的に引きずられ
    正解が 1・2 位 → 4・6 位に落ち、186 倍遅くなった（実測）。"""
    payload = {"model": model, "prompt": prompt, "stream": False,
               "options": {"temperature": 0, "num_predict": 1},
               "logprobs": True, "top_logprobs": min(top_logprobs, 20)}
    if no_think:
        payload["think"] = False
    req = urllib.request.Request(OLLAMA_HOST + "/api/generate", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=CFG["ollama"]["timeout_sec"]) as r:
            data = json.loads(r.read().decode())
    except urllib.error.HTTPError:
        if not no_think:
            raise
        return first_token_logprobs(prompt, model, top_logprobs, no_think=False)
    if "error" in data:
        raise RuntimeError(str(data["error"])[:200])
    lp = (data.get("logprobs") or [None])[0]
    if not lp:
        raise RuntimeError("logprobs が返りません。Ollama v0.12.11 以降が必要です。")
    out = {}
    if lp.get("token") is not None:
        out[lp["token"].strip()] = lp["logprob"]
    for alt in (lp.get("top_logprobs") or []):
        t, v = alt.get("token"), alt.get("logprob")
        if t is None:
            for k, vv in alt.items():
                out.setdefault(str(k).strip(), vv)
        elif v is not None:
            out.setdefault(str(t).strip(), v)
    return lp.get("token"), out


M = CFG["models"]
STAGE_MODEL = {s: (M.get(s) or M["default"]) for s in ("stage0", "stage1", "stage2")}

AVAILABLE = list_local_models()
_sizes = {m["name"]: m["size_gb"] for m in AVAILABLE}
for s, m in STAGE_MODEL.items():
    note = f"{_sizes[m]:.1f}GB" if m in _sizes else ("⚠ 一覧にありません（ollama pull が必要かも）" if AVAILABLE else "")
    print(f"  {s}: {m:<24} {note}")


def prepare_stage(stage):
    """その段階で使わないモデルを降ろす。2 つ載ったままだと載せ替えで 30 倍遅くなる。"""
    need = STAGE_MODEL[stage]
    for name, _ in resident_models():
        if name != need and name in STAGE_MODEL.values():
            unload(name)

## ③ プロンプト（全段階で同じ考え方）

どの段階も、先頭に同じ見出しを置きます。

```
Disease: <疾患名>
Disease mechanism:
- <段階 1>
- <段階 2>
```

その後に段階ごとの問いが続きます。段階0 は遺伝子記号のみ、段階1・2 は選択肢に
「記号（蛋白質名）（aliases: 別名）」を表示します。

段階0 で遺伝子名を**末尾に**置くのは速度のためです。そこまでが全候補で同一になり、
KV キャッシュがそのまま効きます（実測 1.8 倍速い）。

In [ ]:
LETTERS = list(string.ascii_uppercase)
NONE = CFG["rating"]["none_label"]    # 「その他」「どちらも関係ない」を表す仮想の遺伝子
FS = CFG["few_shot"]
DISP = CFG["display"]


def mechanism_block(disease, steps):
    return f"Disease: {disease}\nDisease mechanism:\n" + "".join(f"- {s}\n" for s in steps)


def label(gene, few_shot=False):
    """選択肢 1 行分の表示。記号（蛋白質名）（aliases: 別名）。

    蛋白質名は同族の区別に効く（群 5 の正解率 25/40 → 34/40）。HGNC 正式名は同族でほぼ同じ
    文言になり悪化する（→ 31/40）ので既定は外す。別名は、よく知られた名前（SLC22A12 = URAT1、
    SLC5A2 = SGLT2）でモデルが遺伝子を取り違えないようにするため。"""
    prot = (FS["protein_names"] if few_shot else PROTEIN_NAMES).get(gene)
    hgnc = None if few_shot else GENE_NAMES.get(gene)
    alts = (FS.get("aliases") or {} if few_shot else ALIASES).get(gene) or []
    parts = []
    if DISP["gene_names"] and hgnc:
        parts.append(hgnc)
    if DISP["protein_names"] and prot:
        parts.append(prot)
    s = gene + (f" ({'; '.join(parts)})" if parts else "")
    if DISP["aliases"] and alts:
        s += f" ({DISP['alias_label']}: {', '.join(alts[:DISP['max_aliases']])})"
    return s


# ---------- 段階0: 二値 ----------
def binary_prefix():
    """遺伝子名の直前までの共通部分。"""
    fs, q = FS["binary"], CFG["stage0"]["question"]
    shots = "".join(mechanism_block(fs["disease"], fs["mechanism"]) + f"{q}\nGene: {g}\nAnswer: {a}\n\n"
                    for g, a in fs["examples"])
    return "Answer Yes or No.\n\n" + shots + mechanism_block(DISEASE, MECHANISM) + f"{q}\nGene: "


BINARY_PREFIX = binary_prefix()


def binary_margin(gene, model):
    """logP(Yes) − logP(No)。上位 20 に入らなかった側は -99。差を取ると、疾患名の長さなどで
    Yes と No が一緒に上下する成分が消える。"""
    _, raw = first_token_logprobs(BINARY_PREFIX + gene + "\nAnswer:", model)
    return raw.get("Yes", -99.0) - raw.get("No", -99.0)


# ---------- 段階1・2: 選択式 ----------
def choice_block(disease, steps, question, labels, exit_text):
    items = list(labels) + ([exit_text] if exit_text else [])
    return (mechanism_block(disease, steps) + question + "\n"
            + "\n".join(f"{L}. {t}" for L, t in zip(LETTERS, items)) + "\nAnswer:")


def build_choice_prompt(genes, stage):
    """→ (プロンプト, {ラベル: 遺伝子 または NONE})。stage は "stage1" か "stage2"。

    見本は本番と同じ形にする。1 対 1 には 1 対 1 の見本（few_shot.pair）を使う。"""
    sc = CFG[stage]
    exit_text = sc["exit_text"] if sc["exit_option"] else None
    q = sc["question"]
    examples = FS["pair"] if stage == "stage2" else FS["choice"]
    shots = ""
    for ex in examples:
        ans = LETTERS[ex["options"].index(ex["answer"])]
        shots += choice_block(ex["disease"], ex["mechanism"], q,
                              [label(g, few_shot=True) for g in ex["options"]], exit_text) + f" {ans}\n\n"
    body = choice_block(DISEASE, MECHANISM, q, [label(g) for g in genes], exit_text)
    mapping = dict(zip(LETTERS, list(genes) + ([NONE] if exit_text else [])))
    return "Answer with a single letter only.\n\n" + shots + body, mapping


def ranker(genes, stage, model, shuffle_labels=True):
    """1 群 → {遺伝子 or NONE: 確率}。確率は群の中で正規化する。

    ラベルが 1 つも返らなければ空を返す。全部を下限値で埋めると「動いた」ように見えて、
    壊れていることに気づけない。"""
    genes = list(genes)
    if shuffle_labels:
        random.shuffle(genes)
    prompt, mapping = build_choice_prompt(genes, stage)
    tok, raw = first_token_logprobs(prompt, model)
    got = {L: raw[L] for L in mapping if L in raw}
    if not got:
        return {"probs": {}, "top": None, "generated": tok, "n_missing": len(mapping)}
    floor = min(got.values()) - 10.0       # 見えなかったラベルは下限に置く
    filled = {L: got.get(L, floor) for L in mapping}
    top_lp = max(filled.values())
    ex = {L: math.exp(v - top_lp) for L, v in filled.items()}
    z = sum(ex.values())
    probs = {mapping[L]: ex[L] / z for L in mapping}
    return {"probs": probs, "top": max(probs, key=probs.get), "generated": tok,
            "n_missing": len(mapping) - len(got)}


# ---------- 実際に送るプロンプトを確認 ----------
_demo = GENES[:CFG["stage1"]["group_size"]]
print("===== 段階0（対象部分）=====")
print(BINARY_PREFIX.split("\n\n")[-1] + f"{_demo[0]}\nAnswer:\n")
print("===== 段階1（対象部分）=====")
print(build_choice_prompt(_demo, "stage1")[0].split("\n\n")[-1] + "\n")
print("===== 段階2（対象部分）=====")
print(build_choice_prompt(_demo[:2], "stage2")[0].split("\n\n")[-1])

## ④ 段階0 — 二値

全候補を 1 個ずつ独立に採点し、`stage0.keep` 個を残します。候補数が `keep` 以下なら採点はしますが切りません。

遺伝子ごとに独立なので、候補リストに何が入っていても各遺伝子のスコアは変わりません。

In [ ]:
def screen(genes, model, cycles):
    total = len(genes)
    step = max(1, -(-total // max(1, cycles)))
    margins, failures = {}, 0
    t0 = t_cycle = time.time()
    best_rate = None
    print(f"段階0  {total} 遺伝子  モデル {model}  （{cycles} サイクル、1 サイクル {step} 件）", flush=True)
    for i, g in enumerate(genes, 1):
        try:
            margins[g] = binary_margin(g, model)
        except Exception as e:
            failures += 1
            if failures <= 3:
                print(f"    {g}: 失敗 {type(e).__name__}: {e}")
        if i % step and i != total:
            continue
        now = time.time()
        done_here = step if i % step == 0 else i % step
        rate = (now - t_cycle) / done_here          # そのサイクルだけの秒/件（通算だと回復が見えない）
        t_cycle = now
        best_rate = rate if best_rate is None else min(best_rate, rate)
        over0 = sum(v > 0 for v in margins.values())
        lead = max(margins, key=margins.get) if margins else "-"
        print(f"  {-(-i // step):>3}/{cycles}  {i:>6}/{total}  {rate:>5.2f}秒/件  経過 {(now - t0) / 60:>5.1f}分"
              f"  残り {(total - i) * rate / 60:>5.1f}分  Yes {over0}  首位 {lead}", flush=True)
        if best_rate and rate > 3 * best_rate and rate > 2:
            print("    ⚠ 最速の 3 倍以上遅い。別のモデルとの載せ替えを `ollama ps` で確認してください。")
    print(f"  {(time.time() - t0) / 60:.1f}分  失敗 {failures}")
    return margins


if run_file("stage0_screen.tsv"):
    _rows = load_rows("stage0_screen.tsv")
    SCREEN = {r["gene"]: float(r["margin"]) for r in _rows}
    SURVIVORS = [r["gene"] for r in _rows if r["passed"] == "1"]
    print(f"段階0 は保存済みを読み込みました（{len(SCREEN)} 件、通過 {len(SURVIVORS)}）。モデルは呼びません。")
else:
    prepare_stage("stage0")
    SCREEN = screen(GENES, STAGE_MODEL["stage0"], CFG["stage0"]["progress_cycles"])
    _order = sorted(SCREEN, key=lambda g: -SCREEN[g])
    SURVIVORS = _order[:CFG["stage0"]["keep"]]
    _pass = set(SURVIVORS)
    save_rows("stage0_screen.tsv", ["rank", "gene", "margin", "passed"],
              [[i, g, round(SCREEN[g], 4), int(g in _pass)] for i, g in enumerate(_order, 1)])
    record_meta(stage0={"model": STAGE_MODEL["stage0"], "scored": len(SCREEN), "kept": len(SURVIVORS),
                        "yes": sum(v > 0 for v in SCREEN.values())})

_order = sorted(SCREEN, key=lambda g: -SCREEN[g])
print(f"\n  {len(SCREEN)} → {len(SURVIVORS)}   Yes 側 {sum(v > 0 for v in SCREEN.values())} 個")
for g in KNOWN_ANSWERS:
    if g in SCREEN:
        print(f"  {g:<10} {_order.index(g) + 1:>5} 位  {SCREEN[g]:+6.1f}  {'通過' if g in SURVIVORS else '⚠ 脱落'}")

## ⑤ 段階1 — 選択式（群＋「その他」）

`stage1.rounds` の小段階ごとに、`group_size` 個ずつの群を何度も出題して平均確率で切ります。
群は山札方式で配るので、どの遺伝子も登場回数の差は最大 1 です。

「その他」を入れるのは、関係する遺伝子を 1 つも含まない群から偽の勝者を出さないためです
（「その他」が正解率を上げるわけではない。囮を揃えて測ると 6/20 対 5/20）。

In [ ]:
def balanced_groups(genes, n, rounds, rng):
    """山札方式。全遺伝子を切って上から配り、尽きたら切り直す。同一群内の重複だけ飛ばす。"""
    n = min(n, len(genes))
    deck, out = [], []
    for _ in range(rounds):
        group = []
        while len(group) < n:
            if not deck:
                deck = list(genes)
                rng.shuffle(deck)
            picked = next((i for i, g in enumerate(deck) if g not in group), None)
            if picked is None:
                deck = []
                continue
            group.append(deck.pop(picked))
        out.append(group)
    return out


def make_plan(n_genes, rounds_cfg, group_size):
    """YAML の rounds（keep と appearances）→ [(出題数, 残す数), ...]。"""
    plan, pool = [], n_genes
    for r in rounds_cfg:
        keep = min(r["keep"], pool)
        plan.append((max(1, round(pool * r["appearances"] / group_size)), keep, r["appearances"], pool))
        pool = keep
    return plan


def preview_plan(plan, sec_per_call):
    total = 0
    print(f"  {'対象':>6} → {'残す':>5}  {'出題':>6}  {'登場/個':>7}  {'時間':>6}")
    for calls, keep, appear, pool in plan:
        total += calls
        print(f"  {pool:>6} → {keep:>5}  {calls:>6}  {appear:>7}  {calls * sec_per_call / 60:>5.0f}分")
        if appear < 3:
            print("     ⚠ 登場 3 回未満では、切る根拠が出題の偶然になります")
        if keep / pool < 0.3:
            print(f"     ⚠ 一度に {100 * (1 - keep / pool):.0f}% を落とします。小段階を増やすほうが安全です")
    print(f"  合計 {total} 回  約 {total * sec_per_call / 60:.0f} 分")


def winnow(genes, model, plan, group_size, seed):
    """→ record（遺伝子ごとの到達段階・平均・登場数）、final_order、matches（BT 用）。"""
    pool, record, matches = list(genes), {}, []
    calls = failures = exits = 0
    print(f"段階1  モデル {model}", flush=True)
    for si, (n_calls, keep, _, _) in enumerate(plan, 1):
        groups = balanced_groups(pool, group_size, n_calls, random.Random(seed + si))
        collected = defaultdict(list)
        t0 = time.time()
        for i, grp in enumerate(groups, 1):
            try:
                res = ranker(grp, "stage1", model)
                calls += 1
            except Exception as e:
                failures += 1
                print(f"  小段階{si} 出題{i}: 失敗 {type(e).__name__}: {e}")
                continue
            if not res["probs"]:
                failures += 1
                continue
            exits += res["top"] == NONE
            for g in grp:
                collected[g].append(res["probs"][g])
            # BT 用には NONE も群の一員として残す（「その他」を仮想の対戦相手として推定するため）
            matches.append((tuple(grp) + ((NONE,) if NONE in res["probs"] else ()), res["probs"]))
            if i % 200 == 0:
                print(f"    小段階{si}: {i}/{len(groups)}  {(time.time() - t0) / 60:.1f}分", flush=True)
        means = {g: statistics.mean(v) for g, v in collected.items()}
        for g in pool:
            record[g] = {"stage": si, "mean": means.get(g), "n": len(collected.get(g, []))}
        pool = sorted(pool, key=lambda g: -means.get(g, float("-inf")))[:keep]
        print(f"  小段階{si}: {len(groups)} 出題 → 残り {len(pool)}", flush=True)
    return {"record": record, "final_order": pool, "matches": matches,
            "calls": calls, "failures": failures, "exits": exits}


def measure_sec_per_call(model, pool, stage, group_size, n=6, warmup=2):
    """1 回の実時間（中央値）。他のモデルと取り合っている間に 1 回だけ 28 秒が混ざったことがあり、
    平均だと見積もりが 40 倍ずれた。結果は run.json に残し、再開時は測り直さない。"""
    cached = read_meta().get("sec_per_call", {}).get(f"{stage}:{model}")
    if cached is not None:
        return cached
    rng, ts = random.Random(12345), []
    for k in range(n):
        grp = rng.sample(list(pool), min(group_size, len(pool)))
        t0 = time.time()
        ranker(grp, stage, model)
        if k >= warmup:
            ts.append(time.time() - t0)
    sec = statistics.median(ts)
    record_meta(sec_per_call={**read_meta().get("sec_per_call", {}), f"{stage}:{model}": round(sec, 3)})
    return sec


S1 = CFG["stage1"]
PLAN = make_plan(len(SURVIVORS), S1["rounds"], S1["group_size"])
_files1 = ("stage1_record.tsv", "stage1_matches.json", "stage1_final.json")

if all(run_file(f) for f in _files1):
    with open(run_file("stage1_matches.json")) as fh:
        _m = [(tuple(x["group"]), x["probs"]) for x in json.load(fh)]
    with open(run_file("stage1_final.json")) as fh:
        _fin = json.load(fh)
    _rec = {r["gene"]: {"stage": int(r["stage"]), "mean": float(r["mean"]) if r["mean"] else None,
                        "n": int(r["n"])} for r in load_rows("stage1_record.tsv")}
    _s1 = read_meta().get("stage1", {})
    WIN = {"record": _rec, "final_order": _fin, "matches": _m, "calls": _s1.get("calls", len(_m)),
           "failures": _s1.get("failures", 0), "exits": _s1.get("exits", 0)}
    print(f"段階1 は保存済みを読み込みました（{len(_m)} 出題、通過 {len(_fin)}）。モデルは呼びません。")
else:
    prepare_stage("stage1")
    _sec = measure_sec_per_call(STAGE_MODEL["stage1"], SURVIVORS, "stage1", S1["group_size"])
    print(f"1 回の実測（中央値）: {_sec:.2f} 秒")
    preview_plan(PLAN, _sec)
    WIN = winnow(SURVIVORS, STAGE_MODEL["stage1"], PLAN, S1["group_size"], S1["seed"])
    save_rows("stage1_record.tsv", ["gene", "stage", "mean", "n", "passed"],
              [[g, r["stage"], "" if r["mean"] is None else round(r["mean"], 6), r["n"],
                int(g in set(WIN["final_order"]))] for g, r in WIN["record"].items()])
    save_json("stage1_matches.json", [{"group": list(g), "probs": {k: round(v, 6) for k, v in p.items()}}
                                      for g, p in WIN["matches"]])
    save_json("stage1_final.json", WIN["final_order"])
    record_meta(stage1={"model": STAGE_MODEL["stage1"], "plan": [p[:2] for p in PLAN], "calls": WIN["calls"],
                        "failures": WIN["failures"], "exits": WIN["exits"], "kept": len(WIN["final_order"])})

print(f"\n  出題 {WIN['calls']}  失敗 {WIN['failures']}  「その他」が 1 位 {WIN['exits']}/{WIN['calls']}")
for g in KNOWN_ANSWERS:
    if g in WIN["record"]:
        _pos = WIN["final_order"].index(g) + 1 if g in WIN["final_order"] else None
        print(f"  {g:<10} 到達 小段階{WIN['record'][g]['stage']}" + (f"  通過 {_pos} 位" if _pos else "  脱落"))

## ⑥ 段階2 — 総当たり（第3の選択肢「どちらも関係ない」）

段階1 を通った遺伝子を 1 対 1 で全ペア戦わせます。A/B を入れ替えて 2 回聞き（`both_ways`）、
選択肢の順番による偏りを打ち消します。

対戦は 1 件ごとに `stage2_matches.jsonl` へ書くので、止めても続きから再開できます。
途中で遅くなったら（直近の速度が最速の `slow_factor` 倍）モデルを降ろして読み直させます。
ただし別のモデルが載っていれば、別プロセスとの取り合いなので降ろさずに止めます。

In [ ]:
def load_checkpoint(path):
    done, matches = set(), []
    if path and os.path.exists(path):
        with open(path) as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:      # 書き込み途中で落ちた最終行
                    continue
                key = tuple(rec["order"])
                if key not in done:
                    done.add(key)
                    matches.append((tuple(rec["group"]), rec["probs"]))
    return done, matches


def round_robin(genes, model, both_ways, checkpoint):
    sc = CFG["stage2"]
    pairs = list(itertools.combinations(genes, 2))
    order = [o for a, b in pairs for o in (((a, b), (b, a)) if both_ways else ((a, b),))]
    done, matches = load_checkpoint(checkpoint)
    todo = [o for o in order if o not in done]
    print(f"段階2  モデル {model}  {len(pairs)} ペア x {2 if both_ways else 1} 方向 = {len(order)} 回", flush=True)
    if done:
        print(f"  保存済みから再開: 済み {len(order) - len(todo)} / 残り {len(todo)}")
    if not todo:
        return matches
    fh = open(checkpoint, "a") if checkpoint else None
    failures = resets = n_win = 0
    t0 = t_win = time.time()
    best, skip = None, True          # 最初の区間はモデルの読み込みを含むので速度の基準に入れない
    try:
        for k, o in enumerate(todo, 1):
            try:
                res = ranker(list(o), "stage2", model, shuffle_labels=False)
            except Exception as e:
                failures += 1                      # 保存しないので再開時に試し直される
                if failures <= 3:
                    print(f"  {o}: 失敗 {type(e).__name__}: {e}")
                res = None
            if res and res["probs"]:
                grp = tuple(o) + ((NONE,) if NONE in res["probs"] else ())
                probs = {g: round(p, 6) for g, p in res["probs"].items()}
                matches.append((grp, probs))
                if fh:
                    fh.write(json.dumps({"order": list(o), "group": list(grp), "probs": probs},
                                        ensure_ascii=False) + "\n")
                    fh.flush()
            n_win += 1
            if k % sc["report_every"] and k != len(todo):
                continue
            now = time.time()
            rate = (now - t_win) / n_win
            n_done = len(order) - len(todo) + k
            print(f"  {n_done:>6}/{len(order)}  {rate:>5.2f}秒/件  経過 {(now - t0) / 60:>5.1f}分"
                  f"  残り {(len(todo) - k) * rate / 60:>5.0f}分", flush=True)
            t_win, n_win = now, 0
            if skip:
                skip = False
                continue
            best = rate if best is None else min(best, rate)
            if k != len(todo) and rate > sc["slow_factor"] * best and rate > sc["slow_min_sec"]:
                others = [n for n, _ in resident_models() if n != model]
                if others:
                    print(f"    ⚠ 最速の {rate / best:.1f} 倍。別のモデル {others} が載っています。"
                          "別プロセスと取り合っているので止めます。相手が終わってから再開してください。")
                    raise RuntimeError("段階2 を中断（別のモデルと VRAM を取り合っている）")
                resets += 1
                if resets > sc["max_resets"]:
                    raise RuntimeError("段階2 を中断（リセットしても速度が戻らない。発熱・他のプロセスを疑う）")
                print(f"    ⚠ 最速の {rate / best:.1f} 倍。{model} を降ろして読み直させます（{resets}/{sc['max_resets']}）")
                unload(model)
                time.sleep(3)
                t_win, skip = time.time(), True
    except (KeyboardInterrupt, RuntimeError):
        print(f"\n  止めました。済んだ対戦は {checkpoint} に保存済み。")
        print(f"  再開は YAML の output.resume_dir に \"{RUN_DIR}\" を入れて全セルを実行。")
        raise
    finally:
        if fh:
            fh.close()
    print(f"  今回 {len(todo)} 回  {(time.time() - t0) / 60:.1f}分  失敗 {failures}  リセット {resets}")
    return matches


FINALISTS = list(WIN["final_order"])
_s2 = read_meta().get("stage2", {})
_ck = os.path.join(RUN_DIR, "stage2_matches.jsonl")
if os.path.exists(_ck) and _s2.get("model") not in (None, STAGE_MODEL["stage2"]):
    raise ValueError(f"保存済みの段階2 は {_s2['model']} の対戦です（いまは {STAGE_MODEL['stage2']}）。")
prepare_stage("stage2")
_n = len(FINALISTS) * (len(FINALISTS) - 1) // 2 * (2 if CFG["stage2"]["both_ways"] else 1)
print(f"総当たり: {len(FINALISTS)} 遺伝子 → {_n} 回")
record_meta(stage2={"model": STAGE_MODEL["stage2"], "n_genes": len(FINALISTS), "calls_planned": _n})
RR = round_robin(FINALISTS, STAGE_MODEL["stage2"], CFG["stage2"]["both_ways"], _ck)

## ⑦ 順位づけ（Bradley-Terry）

段階1 と段階2 の出題を**まとめて 1 つのモデルに食わせます**。どちらも「群と、選択肢ごとの確率」という
同じ形なので、追加の推論なしに順位が出ます。

**「その他」「どちらも関係ない」は仮想の遺伝子 NONE として一緒に強さを推定します。**
NONE より弱い遺伝子は、「どちらも関係ない」に負けることが多かった遺伝子、つまり
**関係ないと判定された遺伝子**です。順位に加えて線引きが出ます。

In [ ]:
def bradley_terry(matches, alpha, iters=500, tol=1e-9):
    """Luce/BT を MM 法で最尤推定する。観測は確率ベクトルなので、勝ち数は小数で数える。

    alpha は強さ 1 の仮想の相手との引き分け。一度も勝てなかった遺伝子が -inf に飛ぶのを防ぐ。"""
    W, sets = defaultdict(float), defaultdict(list)
    for grp, probs in matches:
        for g in grp:
            W[g] += probs.get(g, 0.0)
            sets[g].append(grp)
    if not W:
        return {}
    pi = {g: 1.0 for g in W}
    for _ in range(iters):
        cache, new = {}, {}
        for g in pi:
            d = 0.0
            for grp in sets[g]:
                s = cache.get(id(grp))
                if s is None:
                    s = cache[id(grp)] = sum(pi[x] for x in grp if x in pi)
                if s > 0:
                    d += 1.0 / s
            d += 2 * alpha / (pi[g] + 1.0)
            new[g] = (W[g] + alpha) / d if d > 0 else pi[g]
        gm = math.exp(statistics.mean(math.log(v) for v in new.values() if v > 0))
        new = {g: v / gm for g, v in new.items()}
        delta = max(abs(new[g] - pi[g]) for g in pi)
        pi = new
        if delta < tol:
            break
    return {g: math.log(v) for g, v in pi.items()}


elo = lambda v: 1500 + 400 * v / math.log(10)
BT = bradley_terry(WIN["matches"] + RR, CFG["rating"]["alpha"])
NONE_RATING = BT.get(NONE)
FINAL = sorted(FINALISTS, key=lambda g: -BT.get(g, float("-inf")))

rows = []
for i, g in enumerate(FINAL, 1):
    rows.append({"rank": i, "gene": g, "elo": round(elo(BT[g]), 1),
                 "relevant": None if NONE_RATING is None else BT[g] > NONE_RATING,
                 "stage0_rank": sorted(SCREEN, key=lambda x: -SCREEN[x]).index(g) + 1,
                 "stage0_margin": round(SCREEN[g], 2),
                 "protein": PROTEIN_NAMES.get(g, ""), "aliases": ",".join(ALIASES.get(g, []))})

if NONE_RATING is not None:
    _n_rel = sum(r["relevant"] for r in rows)
    print(f"「関係ない」の線（NONE）: elo {elo(NONE_RATING):.0f}  → 関係ありと判定 {_n_rel} / {len(rows)}")
if HAVE_PANDAS:
    _df = pd.DataFrame(rows)
    try:
        display(_df)
    except NameError:
        print(_df.to_string(index=False))
else:
    for r in rows:
        print(r)

print("\n答え合わせ:")
for g in KNOWN_ANSWERS:
    if g in FINAL:
        r = rows[FINAL.index(g)]
        print(f"  {g:<10} {r['rank']:>3} 位 / {len(FINAL)}  elo {r['elo']:.0f}  {'関係あり' if r['relevant'] else '関係なし'}")
    elif g in WIN["record"]:
        print(f"  {g:<10} 段階1 で脱落")
    elif g in SCREEN:
        print(f"  {g:<10} 段階0 で脱落（{sorted(SCREEN, key=lambda x: -SCREEN[x]).index(g) + 1} 位）")

save_rows("final_ranking.tsv", list(rows[0].keys()), [list(r.values()) for r in rows])
save_rows("bradley_terry.tsv", ["gene", "log_strength", "elo"],
          [[g, round(v, 6), round(elo(v), 1)] for g, v in sorted(BT.items(), key=lambda kv: -kv[1])])
record_meta(finished=time.strftime("%Y-%m-%d %H:%M:%S"), n_relevant=None if NONE_RATING is None else _n_rel)
print(f"\nこの実行の成果物: {RUN_DIR}/")
for f in sorted(os.listdir(RUN_DIR)):
    print(f"  {f}")

## 数字を信じる前に

- **段階0 で落ちた遺伝子は「関係ない」と判定されたのではありません。**上位 `keep` 個に入らなかっただけです。
- **メカニズムの書き方で結果が変わります。**各段階は「変えられる要因」を主語にしてください。
  「シスチンが濃縮して結石になる」では AVPR2 に届かず、「腎臓が尿を濃縮するほど…」なら届きました。
- **腎臓の輸送体のように、臓器や系統が近い遺伝子は Yes になりやすい**です。シスチン尿症では、
  無関係な尿酸の輸送体 SLC22A12 がどの聞き方でも上位に残りました。上位は仮説の種として読み、
  そのまま信じないでください。
- **温度は 0 です。**同じ設定で回し直しても同じ答えが返ります。再現性の確認には、候補の並びを変えてください。